# STR Tract Analysis

Analysis of STR (Short Tandem Repeat) tracts detected by RPTRF tool and comparison with de novo assembly statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from Bio import SeqIO
from collections import defaultdict

In [ ]:
# Define paths
DATA_DIR = Path("/home/peterkad/pkadmaster/data")
SAMPLE_NAME = "tr"  # update per sample
REPEAT_REGIONS_DIR = DATA_DIR /SAMPLE_NAME/ "repeatregions" 
FASTA_FILE = Path("/home/peterkad/pkadmaster/data/ph/ph_diploid.fa")

In [ ]:
def parse_str_file(file_path, include_homopolymer_info=False):
    """
    Parse a single STR tract file and return list of tract information.
    
    Args:
        file_path: Path to the STR tract file
        include_homopolymer_info: If True, also identify homopolymers
        
    Returns:
        If include_homopolymer_info=False: List of tuples (start, end, length)
        If include_homopolymer_info=True: List of tuples (start, end, length, is_homopolymer)
    """
    tracts = []
    with open(file_path, "r") as file:
        for line in file:
            # Skip header lines
            if (
                line.startswith("*")
                or line.strip() == ""
                or line.startswith("Start")
            ):
                continue
            
            parts = line.split()
            if len(parts) >= 3:
                try:
                    start = int(parts[0])
                    end = int(parts[1])
                    length = int(parts[2])
                    
                    if include_homopolymer_info and len(parts) >= 4:
                        # Extract motif sequence from column 3 (format: "Size(Sequence)")
                        motif_info = parts[3]
                        is_homopolymer = False
                        
                        # Check if motif info contains parentheses
                        if "(" in motif_info and ")" in motif_info:
                            try:
                                # Extract sequence from parentheses
                                _, motif_seq = motif_info.split("(", 1)
                                motif_seq = motif_seq.rstrip(")")
                                
                                # Check if it's a homopolymer (all characters are the same)
                                if len(motif_seq) > 0 and len(set(motif_seq)) == 1:
                                    is_homopolymer = True
                            except:
                                pass
                        
                        tracts.append((start, end, length, is_homopolymer))
                    else:
                        tracts.append((start, end, length))
                except ValueError:
                    # Skip lines that can't be parsed
                    continue
    
    return tracts

In [ ]:
# Find all result-*.txt files in repeatregions folder
str_files = sorted(REPEAT_REGIONS_DIR.glob("result-*.txt"))
print(f"Found {len(str_files)} STR tract files")

In [ ]:
# Parse all STR tract files (with homopolymer info)
all_tracts = []
all_tracts_with_hp = []
tracts_per_file = {}

for str_file in str_files:
    tracts = parse_str_file(str_file, include_homopolymer_info=False)
    tracts_with_hp = parse_str_file(str_file, include_homopolymer_info=True)
    all_tracts.extend(tracts)
    all_tracts_with_hp.extend(tracts_with_hp)
    tracts_per_file[str_file.name] = len(tracts)

print(f"Total STR tracts found: {len(all_tracts)}")

In [ ]:
# Calculate total bases in STR tracts
tract_lengths = [tract[2] for tract in all_tracts]  # Extract length (3rd element)
total_str_bases = sum(tract_lengths)

print(f"Total bases in STR tracts: {total_str_bases:,}")
print(f"Average tract length: {np.mean(tract_lengths):.2f} bases")
print(f"Median tract length: {np.median(tract_lengths):.2f} bases")

In [ ]:
def count_assembly_bases(fasta_path):
    """
    Read FASTA file and return total base count and per-contig sizes.
    
    Args:
        fasta_path: Path to FASTA file
        
    Returns:
        Tuple of (total_bases, contig_sizes_dict)
    """
    total_bases = 0
    contig_sizes = {}
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq_length = len(record.seq)
        total_bases += seq_length
        contig_sizes[record.id] = seq_length
    
    return total_bases, contig_sizes

In [ ]:
# Read FASTA file and calculate total assembly size
total_assembly_bases, contig_sizes = count_assembly_bases(FASTA_FILE)

print(f"Total assembly size: {total_assembly_bases:,} bases")
print(f"Number of contigs: {len(contig_sizes)}")
print(f"Average contig size: {np.mean(list(contig_sizes.values())):,.2f} bases")
print(f"Median contig size: {np.median(list(contig_sizes.values())):,.2f} bases")

In [ ]:
# Calculate descriptive statistics
str_coverage_percent = (total_str_bases / total_assembly_bases) * 100

# Calculate bp per 1 repeat region: average number of reference basepairs per repeat region
# This tells us the spacing/density of repeat regions in the genome
bp_per_repeat_region = total_assembly_bases / len(all_tracts)

stats = {
    "Total STR tracts": len(all_tracts),
    "Total STR bases": f"{total_str_bases:,}",
    "Total assembly bases": f"{total_assembly_bases:,}",
    "STR coverage (%)": f"{str_coverage_percent:.4f}",
    "bp per 1 repeat region": f"{bp_per_repeat_region:.2f}",
    "Mean tract length": f"{np.mean(tract_lengths):.2f}",
    "Median tract length": f"{np.median(tract_lengths):.2f}",
    "Min tract length": f"{np.min(tract_lengths)}",
    "Max tract length": f"{np.max(tract_lengths)}",
    "Std dev tract length": f"{np.std(tract_lengths):.2f}",
    "Number of files processed": len(str_files)
}

# Create summary DataFrame
summary_df = pd.DataFrame([stats]).T
summary_df.columns = ["Value"]
print("\n=== Summary Statistics ===")
print(summary_df.to_string())

## Homopolymer content within repeat regions

This section quantifies how many RPTRF tracts are homopolymers and how much repeat coverage remains when homopolymers are excluded.

In [ ]:
# Homopolymer summary from RPTRF tracts
hp_tracts = [t for t in all_tracts_with_hp if t[3]]
non_hp_tracts = [t for t in all_tracts_with_hp if not t[3]]

hp_bases = sum(t[2] for t in hp_tracts)
non_hp_bases = sum(t[2] for t in non_hp_tracts)

hp_region_count = len(hp_tracts)
non_hp_region_count = len(non_hp_tracts)

hp_pct_of_repeat_bases = (hp_bases / total_str_bases * 100) if total_str_bases else 0.0
hp_pct_of_regions = (hp_region_count / len(all_tracts_with_hp) * 100) if all_tracts_with_hp else 0.0

print(f"Homopolymer tracts: {hp_region_count:,} regions")
print(f"Homopolymer bases: {hp_bases:,} bp")
print(f"Homopolymers (% of repeat bases): {hp_pct_of_repeat_bases:.2f}%")
print(f"Homopolymers (% of repeat regions): {hp_pct_of_regions:.2f}%")

print("\nRepeat regions excluding homopolymers:")
print(f"  Bases: {non_hp_bases:,} bp")
print(f"  Regions: {non_hp_region_count:,} regions")

**Report text helper (fill in with the printed values):**

Repeat region identification  
A modified version of the RPTRF tool was able to detect `{total_str_bases:,} bp` of tandem repeats spanning `{len(all_tracts):,}` repeat regions in the de novo assembly reference genome consisting of `{total_assembly_bases:,} bp`. This corresponds to `{str_coverage_percent:.2f}%` of basepairs being classified as residing in a repeat region, with a repeat region occurring on average about once per `{bp_per_repeat_region:.0f} bp`. The repeat regions consist of `{hp_pct_of_repeat_bases:.2f}%` homopolymer tracts (by bases, `{hp_pct_of_regions:.2f}%` by region count), which, when excluded, leaves `{non_hp_bases:,} bp` in `{non_hp_region_count:,}` regions.

In [ ]:
# Create histogram of tract lengths
plt.figure(figsize=(12, 6))
plt.hist(tract_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Tract Length (bases)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of STR Tract Lengths', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print quartiles
q25, q50, q75 = np.percentile(tract_lengths, [25, 50, 75])
print(f"\nTract Length Quartiles:")
print(f"  25th percentile: {q25:.2f} bases")
print(f"  50th percentile (median): {q50:.2f} bases")
print(f"  75th percentile: {q75:.2f} bases")
print(f"  IQR: {q75 - q25:.2f} bases")

In [ ]:
# Create a log-scale histogram for better visualization of the distribution
plt.figure(figsize=(12, 6))
plt.hist(tract_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Tract Length (bases)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of STR Tract Lengths (Log Scale)', fontsize=14, fontweight='bold')
plt.yscale('log')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create DataFrame for tract data
tract_df = pd.DataFrame(all_tracts, columns=['Start', 'End', 'Length'])

# Display summary statistics table
print("=== Detailed Tract Statistics ===")
print(tract_df['Length'].describe())

In [ ]:
# Visualize tracts per file distribution
tracts_per_file_df = pd.DataFrame(list(tracts_per_file.items()), columns=['File', 'Tract_Count'])
tracts_per_file_df = tracts_per_file_df.sort_values('Tract_Count', ascending=False)

plt.figure(figsize=(14, 6))
plt.bar(range(len(tracts_per_file_df)), tracts_per_file_df['Tract_Count'], alpha=0.7)
plt.xlabel('File Index (sorted by tract count)', fontsize=12)
plt.ylabel('Number of Tracts', fontsize=12)
plt.title('Number of STR Tracts per File', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFiles with most tracts:")
print(tracts_per_file_df.head(10).to_string(index=False))
print(f"\nFiles with fewest tracts:")
print(tracts_per_file_df.tail(10).to_string(index=False))

# Report Addendum

This section documents project results, including DNM rate, indel type relationships, and genomic patterns. Update the paths below to point at the processed outputs for your run.

In [ ]:
# Report configuration
# Update OUTPUT_BASE and SAMPLE_NAME to point at your run outputs
OUTPUT_BASE = Path("/home/peterkad/pkadmaster/indel_scanner/results")
SAMPLE_NAME = "ph"

# Optional: try a few fallback locations if OUTPUT_BASE does not exist
CANDIDATE_BASES = [
    OUTPUT_BASE,
    Path("results"),
    Path("output"),
]

for candidate in CANDIDATE_BASES:
    if candidate.exists():
        OUTPUT_BASE = candidate
        break

sample_dir = OUTPUT_BASE / SAMPLE_NAME
if not sample_dir.exists():
    print(f"Missing sample dir: {sample_dir}")
    run_dir = None
else:
    run_dirs = sorted([p for p in sample_dir.iterdir() if p.is_dir()])
    run_dir = run_dirs[-1] if run_dirs else None

print(f"Using OUTPUT_BASE: {OUTPUT_BASE.resolve()}")
print(f"Using SAMPLE_NAME: {SAMPLE_NAME}")
print(f"Using RUN_DIR: {run_dir}")
RUN_LABEL = run_dir.name if run_dir else "unknown"

In [ ]:
def read_tsv(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists():
        print(f"Missing: {path}")
        return None
    return pd.read_csv(path, sep="\t")

if run_dir is None:
    final_passed_indels_path = None
    per_type_frequency_path = None
else:
    final_passed_indels_path = run_dir / "processed" / "final_passed_indels.tsv"
    per_type_frequency_path = run_dir / "per_type_mutation_frequency.tsv"

passed_df = read_tsv(final_passed_indels_path) if final_passed_indels_path else None
per_type_frequency_df = read_tsv(per_type_frequency_path) if per_type_frequency_path else None

In [ ]:
GROUP_DISTANCE_BP = 5000

def filter_grouped_indels(df: pd.DataFrame, distance_bp: int) -> pd.DataFrame:
    filtered = []
    for contig, group in df.sort_values(["contig", "ref_position"]).groupby("contig"):
        positions = group["ref_position"].to_numpy()
        prev_dist = np.diff(positions, prepend=np.nan)
        next_dist = np.diff(positions, append=np.nan)
        keep_mask = (np.isnan(prev_dist) | (prev_dist > distance_bp)) & (
            np.isnan(next_dist) | (next_dist > distance_bp)
        )
        filtered.append(group[keep_mask])
    return pd.concat(filtered, ignore_index=True) if filtered else df

def assign_bin(length: int) -> str:
    bins = [
        {"label": "indel_1bp", "min": 1, "max": 1},
        {"label": "indel_2_3bp", "min": 2, "max": 3},
        {"label": "indel_4_10bp", "min": 4, "max": 10},
    ]
    for bin_cfg in bins:
        if bin_cfg["min"] <= length <= bin_cfg["max"]:
            return bin_cfg["label"]
    return "indel_gt_10bp"

def get_str_class(df: pd.DataFrame) -> pd.Series:
    if {"in_STR_region", "in_STR"}.issubset(df.columns):
        return np.where(df["in_STR_region"] & df["in_STR"], "STR_motif", "non_STR")
    if "in_STR" in df.columns:
        return np.where(df["in_STR"], "STR", "non_STR")
    return pd.Series(["unknown"] * len(df), index=df.index)

insertions_df = None
deletions_df = None

if passed_df is not None and "ref_position" in passed_df.columns and "contig" in passed_df.columns:
    raw_df = read_tsv(final_passed_indels_path) if final_passed_indels_path else passed_df.copy()
    if raw_df is None or raw_df.empty:
        raw_df = passed_df.copy()

    if "type" in raw_df.columns:
        before_all = len(raw_df)
        raw_df = raw_df[raw_df["type"].isin(["ins", "del"])].copy()
        if len(raw_df) != before_all:
            print(f"Excluding SNPs from report: {before_all:,} -> {len(raw_df):,}")

    before_count = len(raw_df)

    before_by_bin = None
    before_by_bin_str = None
    if "type" in raw_df.columns and "length" in raw_df.columns:
        before_by_bin = (
            raw_df.assign(bin=raw_df["length"].apply(assign_bin))
            .groupby(["type", "bin"]) 
            .size()
        )
        before_by_bin_str = (
            raw_df.assign(
                bin=raw_df["length"].apply(assign_bin),
                str_class=get_str_class(raw_df),
            )
            .groupby(["type", "bin", "str_class"]) 
            .size()
        )

    passed_df = filter_grouped_indels(raw_df, GROUP_DISTANCE_BP)
    after_count = len(passed_df)

    after_by_bin = None
    after_by_bin_str = None
    if "type" in passed_df.columns and "length" in passed_df.columns:
        after_by_bin = (
            passed_df.assign(bin=passed_df["length"].apply(assign_bin))
            .groupby(["type", "bin"]) 
            .size()
        )
        after_by_bin_str = (
            passed_df.assign(
                bin=passed_df["length"].apply(assign_bin),
                str_class=get_str_class(passed_df),
            )
            .groupby(["type", "bin", "str_class"]) 
            .size()
        )

    if before_by_bin_str is not None and after_by_bin_str is not None:
        print("Grouped indel filter by STR class:")
        all_keys = sorted(set(before_by_bin_str.index) | set(after_by_bin_str.index))
        for t, bin_label, str_class in all_keys:
            print(
                f"  {t} {bin_label} {str_class}: "
                f"{before_by_bin_str.get((t, bin_label, str_class), 0):,} -> "
                f"{after_by_bin_str.get((t, bin_label, str_class), 0):,}"
            )

        before_df = before_by_bin_str.reset_index(name="count")
        before_df["stage"] = "before"
        after_df = after_by_bin_str.reset_index(name="count")
        after_df["stage"] = "after"
        plot_df = pd.concat([before_df, after_df], ignore_index=True)
        plot_df["group"] = plot_df["bin"] + " | " + plot_df["str_class"]

        plt.figure(figsize=(10, 4))
        sns.barplot(
            data=plot_df,
            x="group",
            y="count",
            hue="stage",
            estimator="sum",
            errorbar=None,
        )
        plt.title("Grouped indel filter changes by bin + STR class")
        plt.ylabel("Count")
        plt.xlabel("Bin | STR class")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

    if "type" in passed_df.columns:
        insertions_df = passed_df[passed_df["type"] == "ins"].copy()
        deletions_df = passed_df[passed_df["type"] == "del"].copy()

if insertions_df is not None:
    print(f"Insertions: {len(insertions_df):,}")
if deletions_df is not None:
    print(f"Deletions: {len(deletions_df):,}")

## DNM Rate

DNM rate is defined as `N_indels / L_interrogated` (see pipeline reporting).

In [ ]:
if passed_df is not None and {"type", "length", "in_STR"}.issubset(passed_df.columns):
    print("DNM rate by type + STR class (after grouped filter):")
    callable_lookup = {}
    if per_type_frequency_df is not None and not per_type_frequency_df.empty:
        callable_lookup = dict(
            zip(per_type_frequency_df["mutation_type"], per_type_frequency_df["callable_bases"])
        )

    temp_df = passed_df.copy()
    temp_df["str_class"] = get_str_class(temp_df)
    if "str_motif_length" in temp_df.columns:
        temp_df["motif_length"] = pd.to_numeric(
            temp_df["str_motif_length"], errors="coerce"
        )
    else:
        temp_df["motif_length"] = pd.NA

    temp_df["mutation_type"] = temp_df.apply(
        lambda r: (
            f"{'ins' if r['type']=='ins' else 'del'}_motif_{int(r['motif_length'])}bp"
            if r["str_class"] == "STR_motif" and pd.notna(r["motif_length"])
            else f"{'ins' if r['type']=='ins' else 'del'}_" + (
                "indel_1bp" if r["length"] == 1 else
                "indel_2_3bp" if 2 <= r["length"] <= 3 else
                "indel_4_10bp" if 4 <= r["length"] <= 10 else
                "indel_gt_10bp"
            )
        ),
        axis=1,
    )

    grouped = temp_df.groupby(["mutation_type", "str_class"]).size().reset_index(name="count")
    grouped = grouped[~grouped["mutation_type"].str.contains("snp", case=False, na=False)]
    grouped["callable_bases"] = grouped["mutation_type"].map(callable_lookup).fillna(0).astype(float)
    grouped["frequency"] = np.where(
        grouped["callable_bases"] > 0,
        grouped["count"] / grouped["callable_bases"],
        0.0,
    )

    str_table = grouped[grouped["str_class"] == "STR_motif"].sort_values(["mutation_type"])
    non_str_table = grouped[grouped["str_class"] != "STR_motif"].sort_values(["mutation_type"])

    print("\nCopy/paste STR expansion/shrinkage table (TSV):")
    print(str_table.to_csv(sep="\t", index=False))
    print("\nCopy/paste non-STR table (TSV):")
    print(non_str_table.to_csv(sep="\t", index=False))
else:
    print("per_type_mutation_frequency.tsv not available; no DNM rate breakdown.")

## Indel Type Relationships

Compare counts, ratios, and length distributions of insertions vs deletions.

In [ ]:
indels_df = passed_df.copy() if passed_df is not None else None

if indels_df is not None and "type" in indels_df.columns:
    counts = indels_df["type"].value_counts()
    ratio = counts.get("ins", 0) / max(counts.get("del", 1), 1)

    if {"in_STR"}.issubset(indels_df.columns) or {"in_STR_region", "in_STR"}.issubset(indels_df.columns):
        plot_counts = (
            indels_df.assign(
                bin=indels_df["length"].apply(assign_bin),
                str_class=get_str_class(indels_df),
            )
            .groupby(["type", "bin", "str_class"]).size().reset_index(name="count")
        )
        plot_counts["group"] = plot_counts["bin"] + " | " + plot_counts["str_class"]

        ratio_parts = []
        for group in sorted(plot_counts["group"].unique()):
            subset = plot_counts[plot_counts["group"] == group].set_index("type")
            sc_ratio = subset.at["ins", "count"] / max(subset.at["del", "count"], 1) if {"ins", "del"}.issubset(subset.index) else 0
            ratio_parts.append(f"{group}={sc_ratio:.2f}")
        ratio_text = ", ".join(ratio_parts)

        ratio_df = plot_counts.pivot_table(
            index="group",
            columns="type",
            values="count",
            aggfunc="sum",
        ).fillna(0)
        ratio_df["total"] = ratio_df["ins"] + ratio_df["del"]
        ratio_df["ins_ratio"] = np.where(ratio_df["total"] > 0, ratio_df["ins"] / ratio_df["total"], 0)
        ratio_df["del_ratio"] = np.where(ratio_df["total"] > 0, ratio_df["del"] / ratio_df["total"], 0)
        ratio_df = ratio_df.reset_index()

        plot_ratio = ratio_df.melt(
            id_vars=["group", "total"],
            value_vars=["ins_ratio", "del_ratio"],
            var_name="type",
            value_name="ratio",
        )
        plot_ratio["type"] = plot_ratio["type"].str.replace("_ratio", "")

        plt.figure(figsize=(10, 4))
        ax = sns.barplot(data=plot_ratio, x="group", y="ratio", hue="type")
        plt.title("Indel ratios by bin + STR class")
        plt.ylabel("Ratio within group")
        plt.xlabel("Bin | STR class")
        plt.xticks(rotation=45, ha="right")

        y_offset = max(plot_ratio["ratio"].max() * 0.02, 0.02)
        totals = ratio_df.set_index("group")["total"].to_dict()
        for idx, label in enumerate(ax.get_xticklabels()):
            group = label.get_text()
            ax.text(
                idx,
                max(plot_ratio[plot_ratio["group"] == group]["ratio"].max(), 0) + y_offset,
                f"{int(totals.get(group, 0)):,}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

        plt.tight_layout()
        plt.show()
    else:
        plt.figure(figsize=(6, 4))
        sns.barplot(x=counts.index, y=counts.values)
        plt.title(f"Indel counts (ins/del ratio: {ratio:.2f})")
        plt.ylabel("Count")
        plt.show()

    plt.figure(figsize=(8, 4))
    sns.histplot(
        data=indels_df,
        x="length",
        hue="type",
        bins=50,
        log_scale=(False, True),
        element="bars",
        alpha=0.6,
        common_norm=False,
    )
    plt.title("Indel length distribution (log scale)")
    plt.xlabel("Length")
    plt.ylabel("Frequency (log)")
    plt.show()

    if {"length", "in_STR"}.issubset(indels_df.columns) or {"length", "in_STR_region", "in_STR"}.issubset(indels_df.columns):
        split_df = indels_df.assign(
            bin=indels_df["length"].apply(assign_bin),
            str_class=get_str_class(indels_df),
        )
        split_counts = (
            split_df.groupby(["type", "bin", "str_class"]).size().reset_index(name="count")
        )
        print("Indel count split by length bin and STR class:")
        print(split_counts.sort_values(["type", "bin", "str_class"]).to_string(index=False))
        for indel_type in sorted(split_counts["type"].unique()):
            subset = split_counts[split_counts["type"] == indel_type]
            plt.figure(figsize=(8, 4))
            sns.barplot(data=subset, x="bin", y="count", hue="str_class")
            plt.title(f"{indel_type} counts by length bin and STR class")
            plt.xlabel("Length bin")
            plt.ylabel("Count")
            plt.xticks(rotation=45)
            plt.show()

    if per_type_frequency_df is not None:
        plot_df = per_type_frequency_df.copy()
        if "mutation_type" in plot_df.columns:
            plot_df = plot_df[~plot_df["mutation_type"].str.contains("snp", case=False, na=False)]
        plt.figure(figsize=(8, 4))
        sns.barplot(
            data=plot_df,
            x="mutation_type",
            y="frequency",
        )
        plt.title("Per-type mutation frequency")
        plt.ylabel("Frequency")
        plt.xticks(rotation=45)
        plt.show()

## Genomic Patterns

Explore how indels distribute across contigs, along contig positions, and in STR vs non-STR contexts.

In [ ]:
if indels_df is not None:
    # Indels per contig
    contig_counts = indels_df["contig"].value_counts().head(20)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=contig_counts.index, y=contig_counts.values)
    plt.title("Top 20 contigs by indel count")
    plt.ylabel("Count")
    plt.xticks(rotation=90)
    plt.show()

    # Normalize by contig size if available
    if "contig_sizes" in globals() and contig_sizes:
        per_mb = (contig_counts / pd.Series(contig_sizes)).dropna() * 1e6
        per_mb = per_mb.sort_values(ascending=False).head(20)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=per_mb.index, y=per_mb.values)
        plt.title("Top 20 contigs by indels per Mb")
        plt.ylabel("Indels per Mb")
        plt.xticks(rotation=90)
        plt.show()

    # Position-binned density for top contigs
    top_contigs = contig_counts.index[:5]
    bin_count = 100
    smooth_window = 5
    for contig in top_contigs:
        subset = indels_df[indels_df["contig"] == contig].copy()
        if subset.empty:
            continue
        contig_len = contig_sizes.get(contig) if "contig_sizes" in globals() else None
        if contig_len is None:
            contig_len = subset["ref_position"].max()
        bins = np.linspace(0, contig_len, bin_count + 1)
        centers = (bins[:-1] + bins[1:]) / 2
        bin_size = bins[1] - bins[0] if len(bins) > 1 else 1

        if "type" in subset.columns:
            subset["str_class"] = get_str_class(subset)
            types = [t for t in ["del", "ins"] if t in subset["type"].unique()]
        else:
            types = ["all"]
            subset["str_class"] = "all"
            subset["type"] = "all"

        fig, axes = plt.subplots(len(types), 1, figsize=(8, 3 * len(types)), sharex=True)
        if len(types) == 1:
            axes = [axes]

        for ax, indel_type in zip(axes, types):
            type_subset = subset[subset["type"] == indel_type]
            for str_class in sorted(type_subset["str_class"].unique()):
                group_subset = type_subset[type_subset["str_class"] == str_class]
                counts, _ = np.histogram(group_subset["ref_position"], bins=bins)
                rate_per_mb = counts / bin_size * 1e6
                smooth_rate = pd.Series(rate_per_mb).rolling(smooth_window, center=True, min_periods=1).mean().to_numpy()
                ax.plot(centers, smooth_rate, label=str_class)

            ax.set_title(f"{indel_type} density (smoothed): {contig}")
            ax.set_ylabel("Indels per Mb")
            ax.legend(fontsize=8)

        axes[-1].set_xlabel("Position")
        plt.tight_layout()
        plt.show()

        # Density split by length bin (new plot)
        if "length" in subset.columns:
            bin_labels = ["indel_1bp", "indel_2_3bp", "indel_4_10bp", "indel_gt_10bp"]
            bin_subset = subset.copy()
            bin_subset["len_bin"] = bin_subset["length"].apply(assign_bin)

            for bin_label in bin_labels:
                bin_rows = bin_subset[bin_subset["len_bin"] == bin_label]
                fig, axes = plt.subplots(len(types), 1, figsize=(8, 3 * len(types)), sharex=True)
                if len(types) == 1:
                    axes = [axes]

                for ax, indel_type in zip(axes, types):
                    type_rows = bin_rows[bin_rows["type"] == indel_type]
                    if type_rows.empty:
                        ax.set_title(f"{bin_label} | {indel_type} density (smoothed): {contig}")
                        ax.set_ylabel("Indels per Mb")
                        continue
                    for str_class in sorted(type_rows["str_class"].unique()):
                        group_rows = type_rows[type_rows["str_class"] == str_class]
                        counts, _ = np.histogram(group_rows["ref_position"], bins=bins)
                        rate_per_mb = counts / bin_size * 1e6
                        smooth_rate = pd.Series(rate_per_mb).rolling(smooth_window, center=True, min_periods=1).mean().to_numpy()
                        ax.plot(centers, smooth_rate, label=str_class)

                    ax.set_title(f"{bin_label} | {indel_type} density (smoothed): {contig}")
                    ax.set_ylabel("Indels per Mb")
                    ax.legend(fontsize=7)

                axes[-1].set_xlabel("Position")
                plt.tight_layout()
                plt.show()

    # STR vs non-STR proportions
    if "in_STR" in indels_df.columns:
        str_counts = indels_df["in_STR"].value_counts()
        plt.figure(figsize=(5, 3))
        sns.barplot(x=str_counts.index.astype(str), y=str_counts.values)
        plt.title("Indels in STR vs non-STR")
        plt.xlabel("in_STR")
        plt.ylabel("Count")
        plt.show()

## Summary of Findings

- **DNM rate**: Reported per type and STR class using `per_type_mutation_frequency.tsv`.
- **Indel composition**: Insertions vs deletions compared by count and length distribution.
- **Genomic patterns**: Hot contigs, position-binned density, and STR vs non-STR proportions.

## Limitations

- Plots depend on the availability of processed TSV outputs and STR annotations.
- If contig sizes or mutation frequency reports are missing, some plots and metrics are approximate or unavailable.